# Information Retrieval and Web Analysis — Final Project Part 3
## Ranking & Filtering

**Serena El-Hajj · Souad Ibrahim El Husseini · Guillermo**  
**GitHub:** https://github.com/Souad-El-Husseini/irwa-search-engine


This notebook is kept in our **GitHub repository**. To run it online, open it with **Google Colab**. The first code cell downloads the whole repository into the cloud session.

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

repo_url = "https://github.com/Souad-El-Husseini/irwa-search-engine.git"
repo_name = "irwa-search-engine"

if "google.colab" in sys.modules:
    repo_root = Path("/content") / repo_name
    if not (repo_root / "project_progress" / "part_2" / "search_engine.py").exists():
        subprocess.run(["git", "clone", "--depth", "1", repo_url, str(repo_root)], check=True)
else:
    # Also works when the repository is opened online through GitHub Codespaces.
    current = Path.cwd().resolve()
    repo_root = next(
        (folder for folder in [current, *current.parents]
         if (folder / "project_progress" / "part_2" / "search_engine.py").exists()),
        None,
    )
    if repo_root is None:
        raise FileNotFoundError("Open from GitHub in Google Colab or in a repository Codespace")

# Install missing course dependencies in this cloud session.
packages = {"pandas": "pandas", "numpy": "numpy", "nltk": "nltk", "gensim": "gensim>=4.3.3"}
missing = [pip_name for module, pip_name in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

import nltk
nltk.download("stopwords", quiet=True)
import pandas as pd
from IPython.display import display

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print("Using GitHub repository:", repo_root)


In [ ]:
from project_progress.part_3.part3_ranking import Part3Ranker, QUERIES

# Reuse the Part 2 data and indexing functions from this same repository.
engine = Part3Ranker.from_repo(repo_root)
print("Number of products:", engine.N)
print("Index vocabulary:", len(engine.df))
print("Indexed fields:", ", ".join(engine.token_fields[next(iter(engine.token_fields))].keys()))

### keep AND retrieval



In [ ]:
# We preserved the retrieval approach implemented in Part 2 
# and verified that the five queries returned the same matching products before applying the new ranking algorithms.

counts = pd.DataFrame([
    {"Query": query, "AND matches": len(engine.candidates(query))}
    for query in QUERIES
])
display(counts)


## 2. TF-IDF + cosine similarity

We implemented TF-IDF with cosine similarity using the existing index and preprocessing from Part 2.
$$\operatorname{cosine}(q,d)= \frac{\sum_t w_{t,q}w_{t,d}}{\sqrt{\sum_t w_{t,q}^2}\sqrt{\sum_t w_{t,d}^2}}$$


In [ ]:
query = QUERIES[0]
print("TF-IDF cosine:", query)
display(engine.results_table(query, "TF-IDF cosine", top_k=10))


## 3. BM25

We implemented BM25 using \(k_1=1.5\) and \(b=0.75\), considering term frequency and document length across our seven indexed fields.

$$\mathrm{BM25}(q,d) = \sum_{t \in q}\ln\left(1+\frac{N-df_t+0.5}{df_t+0.5}\right) \frac{f(t,d)(k_1+1)}{f(t,d)+k_1(1-b+b|d|/\mathrm{avgdl})}$$

A term repeated many times should not keep gaining influence indefinitely. Document length is the token count across the seven indexed fields from Part 2.

In [ ]:
print("BM25:", query)
display(engine.results_table(query, "BM25", top_k=10))


## 4. Our custom score: Fashion Shopping Score

We developed a custom ranking score combining TF-IDF cosine similarity (60%), title matching (15%), product rating (10%), discount (5%), and stock availability (10%). The weights prioritize query relevance while also considering shopping-related factors.

$$S(q,d)=0.60C(q,d)+0.15T(q,d)+0.10R(d)+0.05D(d)+0.10A(d)$$

- `C`: TF-IDF cosine similarity in `[0,1]`
- `T`: fraction of distinct processed query terms appearing in the product **title**
- `R`: product rating divided by 5 (missing rating receives neutral `0.5`)
- `D`: discount fraction (missing discount receives `0`)
- `A`: in-stock indicator (`1` in stock, `0` out of stock, `0.5` unknown)

The weights add to **1.00**. 

In [ ]:
print("Custom Fashion Shopping Score:", query)
display(engine.results_table(query, "Custom", top_k=10))


## 5. Compare the three lexical ranking approaches

We compared the top 10 results from TF-IDF cosine, BM25, and our custom score to identify differences in product ranking.

In [ ]:
comparison = pd.DataFrame({
    "TF-IDF cosine": [pid for pid, score in engine.tfidf_cosine(query)[:10]],
    "BM25": [pid for pid, score in engine.bm25(query)[:10]],
    "Custom": [pid for pid, score in engine.custom(query)[:10]],
})
comparison.index = range(1, len(comparison) + 1)
comparison.index.name = "Rank"
display(comparison)


## 6. Word2Vec + cosine similarity
We trained a Word2Vec model on our preprocessed product data using gensim. We averaged the word vectors to represent each query and product, then applied cosine similarity to rank the products retrieved through AND search.

$$v_{\mathrm{text}}=\frac{1}{m}\sum_{i=1}^{m}v_{w_i}$$


In [ ]:
# Run once: learning Word2Vec on the 28,080-product corpus.
model = engine.train_word2vec(vector_size=100, window=5, epochs=10, seed=42)
print("Word2Vec vocabulary:", len(model.wv))
print("Word vector size:", model.vector_size)


## 7. Required output: Word2Vec top 20 for **each of the five Part 2 queries**

We applied Word2Vec cosine similarity to our five queries from Part 2 and displayed the top 20 ranked products for each query, where available.

In [ ]:
word2vec_tables = []
for query_id, query in enumerate(QUERIES, 1):
    table = engine.results_table(query, "Word2Vec cosine", top_k=20)
    table.insert(0, "query", query)
    table.insert(0, "query_id", query_id)
    word2vec_tables.append(table)
    print(f"\nQUERY {query_id}: {query} | {len(engine.candidates(query))} AND matches")
    display(table)


## 8. Save reproducible results

We exported the ranking results, query counts, and model information to the results/part3/ folder.

In [ ]:
import json

output = repo_root / "results" / "part3"
output.mkdir(parents=True, exist_ok=True)
rows = []
for qid, q in enumerate(QUERIES, 1):
    for method in ["TF-IDF cosine", "BM25", "Custom", "Word2Vec cosine"]:
        tab = engine.results_table(q, method, top_k=20)
        tab.insert(0, "ranking_method", method)
        tab.insert(0, "query", q)
        tab.insert(0, "query_id", qid)
        rows.append(tab)
all_top20 = pd.concat(rows, ignore_index=True)
all_top20.to_csv(output / "part3_top20_all_methods.csv", index=False)
all_top20[all_top20.ranking_method == "Word2Vec cosine"].to_csv(
    output / "word2vec_top20_five_queries.csv", index=False)
counts.rename(columns={"Query": "query", "AND matches": "and_matches"}).to_csv(
    output / "query_counts.csv", index=False)
model.save(str(output / "word2vec.model"))
(output / "methodology.json").write_text(json.dumps({
    "corpus_size": engine.N, "vocabulary_size": len(engine.df),
    "query_semantics": "Part 2 Boolean AND",
    "methods": ["TF-IDF cosine", "BM25", "Custom", "Word2Vec cosine"],
    "bm25": {"k1": 1.5, "b": 0.75},
    "custom_weights": [0.60, 0.15, 0.10, 0.05, 0.10],
    "word2vec": {"size": 100, "window": 5, "epochs": 10, "seed": 42}
}, indent=2), encoding="utf-8")
print("Results saved to:", output)


## 9. Alternative representation: Doc2Vec or Sentence Embeddings?

We considered Doc2Vec and sentence embeddings as alternatives to Word2Vec. Doc2Vec learns document representations directly, while sentence embeddings can better capture context and meaning. We believe pretrained sentence embeddings could improve our search results, although they require more computational resources.

## 10. Download the results to add to GitHub (Colab only)

We exported our Part 3 results for submission to GitHub.

In [ ]:
import shutil

results_folder = repo_root / "results" / "part3"
zip_path = shutil.make_archive("/content/irwa_part3_results" if "google.colab" in sys.modules
                               else str(repo_root / "irwa_part3_results"),
                               "zip", root_dir=results_folder)
print("Created:", zip_path)

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(zip_path)
else:
    print("Open the ZIP in the current cloud workspace and add its outputs to GitHub.")